# Setup & Imports

In [1]:
# ==============================================================================
# 1. Setup & Imports
# ==============================================================================
import yaml
from copy import deepcopy
from pathlib import Path

CONFIG_FILE_NAME = "campaign_config.yaml"


def merge_config_layer(base, override):
    """Merge one config layer into another, appending parameter_sets."""
    result = deepcopy(base)

    for key, value in override.items():
        if key == "parameter_sets":
            existing = result.get(key, []) or []
            additions = deepcopy(value) if value else []
            result[key] = [*existing, *additions]
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = merge_config_layer(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


def resolve_simulation_config(campaign_config, sim_id):
    """Merge campaign defaults, mode defaults, and simulation overrides."""
    simulations = campaign_config.get("simulations", [])
    sim_entry = next((entry for entry in simulations if entry.get("id") == sim_id), None)
    if sim_entry is None:
        raise KeyError(f"Simulation '{sim_id}' was not found in the campaign config.")

    merged_config = {}
    defaults = campaign_config.get("simulation_defaults", {})
    mode_defaults = campaign_config.get("mode_defaults", {}).get(sim_entry.get("mode"), {})

    for layer in (defaults, mode_defaults, sim_entry):
        merged_config = merge_config_layer(merged_config, layer)

    return merged_config


# Resolve paths
# __file__ is not available in Jupyter notebooks; fall back to Path().resolve()
# which returns the notebook's working directory (its own folder when run in VS Code).
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path().resolve()

CAMPAIGN_DIR = SCRIPT_DIR
PARAMS_DIR = CAMPAIGN_DIR / "parameters"
OPT_CONFIG_DIR = CAMPAIGN_DIR / "optimization_config"

# ── AnimaKinetica imports ──────────────────────────────────────────────────────
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation_data import SimulationData

from animakinetica.models.kinetics import Kinetics
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")
print(f"  Script dir    : {SCRIPT_DIR}")
print(f"  Parameters dir: {PARAMS_DIR}")
print(f"  Opt config dir: {OPT_CONFIG_DIR}")
print(f"  Config file   : {CONFIG_FILE_NAME}")


✓ Imports successful
  Script dir    : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI
  Parameters dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\parameters
  Opt config dir: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\optimization_config
  Config file   : campaign_config.yaml


In [2]:
# ==============================================================================
# 2. Load Campaign Config & Parameters
#
# ▶ SET SIM_ID MANUALLY to the simulation you want to troubleshoot.
#   Valid IDs are the "id" fields in campaign_config.yaml (e.g. "batch_E").
# ==============================================================================
SIM_ID = "fedbatch_gsp"   # <── change this to the simulation you want to inspect
print(f"\n Simulation ID: {SIM_ID}")


 Simulation ID: fedbatch_gsp


In [3]:
# ── Load campaign configuration ────────────────────────────────────────────────
config_path = CAMPAIGN_DIR / CONFIG_FILE_NAME
with open(config_path, encoding="utf-8") as f:
    campaign_config = yaml.safe_load(f)

# Resolve campaign defaults, mode defaults, and simulation overrides
sim_config = resolve_simulation_config(campaign_config, SIM_ID)
opt_settings = campaign_config["optimization_settings"]

# ── Simulation settings ────────────────────────────────────────────────────────
# Convert string names from YAML to enum objects using EnumClass["NAME"] lookup.
# Optional strategies (null in YAML → None in Python) are handled with a conditional.
MODE = OperatingMode[sim_config["mode"]]
KINETICS = Kinetics[sim_config["kinetics"]]
STOICHIOMETRY = StoichiometryMode[sim_config["stoichiometry"]] if sim_config.get("stoichiometry") else None
FEED_STRATEGY = FeedStrategy[sim_config["feed_strategy"]] if sim_config["feed_strategy"] else None
PERFUSION_STRATEGY = (
    PerfusionStrategy[sim_config["perfusion_strategy"]]
    if sim_config["perfusion_strategy"]
    else None
)
SOLVER_METHOD = sim_config.get("solver_method", "LSODA")

# ── Parameter file paths ───────────────────────────────────────────────────────
PARAM_FILES = [
    PARAMS_DIR / Path(entry).with_suffix(".yaml")
    for entry in sim_config["parameter_sets"]
]

# ── Optimization settings ──────────────────────────────────────────────────────
if sim_config["optimize"]:
    OBJECTIVES = opt_settings["objectives"]
    WEIGHTS = [opt_settings["objective_weights"][obj] for obj in OBJECTIVES]
    SEED = opt_settings["seed"]
    WORKERS = opt_settings["workers"]
    TIMEOUT = opt_settings.get("timeout", 5.0)
    OPT_CONFIG_FILE = OPT_CONFIG_DIR / f"{sim_config['opt_config']}_optimization.yaml"

# ── Export filename ────────────────────────────────────────────────────────────
EXPORT_FILENAME = opt_settings["filename"].replace("{simulation_id}", SIM_ID)

print(f"✓ Campaign config loaded: {SIM_ID} — {sim_config['label']}")
print(f"  Mode               : {MODE}")
print(f"  Kinetics           : {KINETICS}")
print(f"  Stoichiometry      : {STOICHIOMETRY}")
print(f"  Feed strategy      : {FEED_STRATEGY}")
print(f"  Perfusion strategy : {PERFUSION_STRATEGY}")
print(f"  Solver method      : {SOLVER_METHOD}")
print(f"  Optimize           : {sim_config['optimize']}")
if sim_config["optimize"]:
    print(f"  Opt config         : {OPT_CONFIG_FILE.name}")
    print(f"  Objectives         : {OBJECTIVES}")
    print(f"  Weights            : {WEIGHTS}")
    print(f"  Seed / Workers     : {SEED} / {WORKERS}")
    print(f"  Timeout            : {TIMEOUT}")
print(f"  Export file        : {EXPORT_FILENAME}")

# ── Load parameters ────────────────────────────────────────────────────────────
params = SimulationParameters.load_parameters([str(p) for p in PARAM_FILES])
print("\n✓ Parameters loaded successfully")
print("  Files:")
for p in PARAM_FILES:
    print(f"    {p.relative_to(PARAMS_DIR)}")


✓ Campaign config loaded: fedbatch_gsp — Fed-Batch | GFI | Glucose Setpoint Feed
  Mode               : Fedbatch
  Kinetics           : Stoichiometric
  Stoichiometry      : StoichiometryMode.DYNAMIC
  Feed strategy      : Glucose Setpoint
  Perfusion strategy : None
  Solver method      : BDF
  Optimize           : True
  Opt config         : F_GSP_optimization.yaml
  Objectives         : ['STY_kg_m3_day', 'medium_use_efficiency_kg_m3', 'nutrient_use_efficiency_kg_kg']
  Weights            : [10.0, 1.0, 1.0]
  Seed / Workers     : 42 / 5
  Timeout            : 10
  Export file        : fedbatch_gsp_opt
Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\parameters\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\parameters\stoichiometry\anab_GFI.ya

In [4]:
# ==============================================================================
# 3. Baseline Simulation
# ==============================================================================

# params["fed_batch.C_Gln_feed"] = 60
# params["fed_batch.C_AA_feed"] = 10

baseline_sim = CellCultureSimulation(
    parameters=params,
    kinetics=KINETICS,
    stoichiometry_mode=STOICHIOMETRY,
    operating_mode=MODE,
    feed_strategy=FEED_STRATEGY,
    perfusion_strategy=PERFUSION_STRATEGY,
    solver_method=SOLVER_METHOD,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,
)

baseline_result = baseline_sim.run_simulation()

print("\n✓ Baseline simulation completed")
print("\nBaseline Performance Metrics:")
for key, value in baseline_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


run_simulation execution time: 470.441 ms

✓ Baseline simulation completed

Baseline Performance Metrics:
  total_DCM_harvested_kg: 3.8065
  STY_kg_m3_day: 0.0059
  medium_use_efficiency_kg_m3: 0.7930
  protein_use_efficiency_kg_kg: 0.3153
  nutrient_use_efficiency_kg_kg: 0.1535
  batch_time_range_h: 0.0000


## Mass Balance Check

In [5]:
baseline_data = baseline_sim.data

print("=" * 70)
print("EXTERNAL BOUNDARY CHON CLOSURE (simulated window)")
print("=" * 70)
boundary_closure = baseline_data.compute_boundary_closure()

rel = {e: v["relative_residual_pct"] for e, v in boundary_closure["elements"].items()}
print(rel)

# Element-level summary
closure_df = baseline_data.build_closure_summary_df(boundary_closure)
print("\nElemental closure summary:")
display(closure_df)

# Max residual
print(f"\nMax absolute residual: {boundary_closure['max_abs_residual']:.6f} mol-atoms")

# Component basis
print("\n" + "-" * 70)
print("Component basis:")
print("-" * 70)
component_df = baseline_data.build_component_specs_df(boundary_closure['component_specs'])
display(component_df)


EXTERNAL BOUNDARY CHON CLOSURE (simulated window)
{'C': -0.0005765306219179053, 'H': -5.194644487331381e-06, 'O': -5.546381715471028e-06, 'N': -0.0017573504993446546}

Elemental closure summary:


,Element,Reactants (mol-atoms),Products (mol-atoms),Residual (mol-atoms),Relative Residual (%)
0,C,129.278233,129.277488,-0.000745,-0.000577
1,H,66836.075625,66836.072153,-0.003472,-0.000005
2,O,33476.416027,33476.414170,-0.001857,-0.000006
3,N,13.252740,13.252507,-0.000233,-0.001757



Max absolute residual: 0.003472 mol-atoms

----------------------------------------------------------------------
Component basis:
----------------------------------------------------------------------


,Component,Side,Description,Amount,Amount Unit,C,H,O,N
0,Fresh glucose in,Input,Total glucose entering with fresh initial char...,12.000000,mol,6.00000,12.000000,6.000000,0.000000
1,Fresh glutamine in,Input,Total glutamine entering with fresh initial ch...,1.800000,mol,5.00000,10.000000,3.000000,2.000000
2,Fresh amino acids in,Input,Total lumped amino-acid pool entering with fre...,6.000000,mol,4.73442,9.982912,1.516180,1.520349
3,Fresh lactate in,Input,Lactate entering from fresh external media only.,0.000000,mol,3.00000,6.000000,3.000000,0.000000
4,Fresh ammonium in,Input,Ammonium entering from fresh external media only.,0.000000,mol,0.00000,3.000000,0.000000,1.000000
5,O2 in,Input,Total oxygen crossing the reactor boundary int...,24.629204,mol,0.00000,0.000000,2.000000,0.000000
6,CO2 in,Input,Total inorganic carbon entering from fresh med...,17.400000,mol,1.00000,0.000000,2.000000,0.000000
7,"Medium, base, and acid water in",Input,Carrier water entering with fresh medium charg...,33305.061037,mol,0.00000,2.000000,1.000000,0.000000
8,DCM in,Input,"Dry biomass entering via inoculum, only with f...",2.471711,Cmol,1.00000,1.641001,0.323461,0.214688
9,Glucose out,Output,Total glucose leaving in external liquid outle...,4.537907,mol,6.00000,12.000000,6.000000,0.000000


In [6]:
# ==============================================================================
# CHON Closure Delta Report — Boundary vs Stoichiometric
# ==============================================================================
# Compares the integrated kinetic-rate predictions (stoichiometric closure)
# against the actual reactor boundary fluxes to surface discrepancies.
print("\n" + "=" * 70)
print("CHON CLOSURE DELTA (boundary − stoichiometric)")
print("=" * 70)

try:
    delta_result = baseline_data.compute_closure_delta()

    # 1. Element-level comparison
    delta_summary_df = SimulationData.build_delta_summary_df(
        delta_result['stoich_closure'], delta_result['boundary_closure']
    )
    print("\nElement-level comparison:")
    display(delta_summary_df)

    # 2. Per-component delta
    print("\n" + "-" * 70)
    print("Per-component delta basis:")
    print("-" * 70)
    delta_component_df = SimulationData.build_delta_component_specs_df(delta_result['component_specs'])
    display(delta_component_df)

    print(f"\nMax absolute delta residual: {delta_result['max_abs_residual']:.6f} mol-atoms")

except ValueError as exc:
    print(f"\n⚠ Delta report unavailable: {exc}")
    print("  (Requires performance_mode=False for stoichiometric rate trajectories.)")


CHON CLOSURE DELTA (boundary − stoichiometric)

Element-level comparison:


,Element,Stoich Reactants (mol-atoms),Stoich Products (mol-atoms),Stoich Residual (mol-atoms),Boundary Reactants (mol-atoms),Boundary Products (mol-atoms),Boundary Residual (mol-atoms),Delta Residual (mol-atoms)
0,C,60.568569,60.567871,-0.000698,129.278233,129.277488,-0.000745,-0.000047
1,H,121.875346,121.873713,-0.001633,66836.075625,66836.072153,-0.003472,-0.001839
2,O,101.593974,101.593512,-0.000462,33476.416027,33476.414170,-0.001857,-0.001395
3,N,5.782825,5.782589,-0.000236,13.252740,13.252507,-0.000233,0.000003



----------------------------------------------------------------------
Per-component delta basis:
----------------------------------------------------------------------


,Component,Side,Description,Stoich Amount,Boundary Net Amount,Delta Amount,Amount Unit,C,H,O,N
0,Glucose consumed delta,Input,Difference between boundary net flux and integ...,7.461821,7.462093,2.717045e-04,mol,6.00000,12.000000,6.000000,0.000000
1,Glutamine consumed delta,Input,Difference between boundary net flux and integ...,1.799800,1.800000,2.001831e-04,mol,5.00000,10.000000,3.000000,2.000000
2,Amino acids consumed delta,Input,Difference between boundary net flux and integ...,1.436003,1.436308,3.045202e-04,mol,4.73442,9.982912,1.516180,1.520349
3,O2 consumed delta,Input,Difference between boundary net flux and integ...,24.623204,24.623204,-8.230749e-09,mol,0.00000,0.000000,2.000000,0.000000
4,Dry biomass delta,Output,Difference between boundary net flux and integ...,19.303074,19.307168,4.093429e-03,Cmol,1.00000,1.641001,0.323461,0.214688
5,Lactate produced delta,Output,Difference between boundary net flux and integ...,4.970146,4.970109,-3.634820e-05,mol,3.00000,6.000000,3.000000,0.000000
6,Ammonium produced delta,Output,Difference between boundary net flux and integ...,1.638447,1.638435,-1.198247e-05,mol,0.00000,3.000000,0.000000,1.000000
7,CO2 produced delta,Output,Difference between boundary net flux and integ...,26.354359,26.354400,4.129774e-05,mol,1.00000,0.000000,2.000000,0.000000
8,H2O produced delta,Output,Difference between boundary net flux and integ...,27.730568,27.730568,1.943334e-11,mol,0.00000,2.000000,1.000000,0.000000



Max absolute delta residual: 0.001839 mol-atoms


In [7]:
baseline_sim.data.plot_kinetics()

In [8]:
baseline_sim.data.plot_stoichiometric_yields()

In [9]:
baseline_sim.data.plot_volume_feed(batch_number=1)

In [10]:
fig_list = baseline_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

## Optimization

In [11]:
# ==============================================================================
# 4. Optimization
#
# Multi-objective optimization using differential evolution. Objectives, weights,
# seed, and workers are all read from campaign_config.yaml → optimization_settings.
# The optimization parameter bounds come from the opt_config file resolved above.
# ==============================================================================
from animakinetica.analysis.optimization import OptimizationManager

# ── Campaign normalization scalars ─────────────────────────────────────────────
# These are the max objective values observed across all baseline simulations and are
# used as a consistent divisor so weights carry identical meaning for every simulation.
NORM_FILE_NAME = "campaign_normalization.yaml"
NORM_FILE = CAMPAIGN_DIR / NORM_FILE_NAME

if NORM_FILE.exists():
    with open(NORM_FILE) as f:
        campaign_norm = yaml.safe_load(f)
    print(f"✓ Loaded campaign normalization from {NORM_FILE.name}")
    for obj, val in campaign_norm.items():
        if isinstance(val, (int, float)):
            print(f"  {obj}: {val:.4g}")
else:
    campaign_norm = None
    print("  ⚠ campaign_normalization.yaml not found; run create_normalization_values.ipynb first.")
    print("    Falling back to per-simulation baseline normalization.")

# ── Initialize OptimizationManager ─────────────────────────────────────────────
opt_manager = OptimizationManager(
    simulation=baseline_sim,
    opt_config_file=str(OPT_CONFIG_FILE),
)

print(f"\n✓ OptimizationManager initialized")
print(f"  Config     : {OPT_CONFIG_FILE.name}")
print(f"  Parameters : {len(opt_manager.opt_config)}")
print("\nParameters to optimize:")
for param, cfg in opt_manager.opt_config.items():
    print(f"  {param}: bounds={cfg['bounds']}")

✓ Loaded campaign normalization from campaign_normalization.yaml
  STY_kg_m3_day: 0.7667
  medium_use_efficiency_kg_m3: 1.059
  nutrient_use_efficiency_kg_kg: 0.1535
Searching specified optimization config: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\optimization_config\F_GSP_optimization.yaml
Loading optimization config from: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_3_dynamic_stoichiometry_GFI\optimization_config\F_GSP_optimization.yaml
Parsing...
Parsed and loaded configuration containing 10 optimization parameters.

✓ OptimizationManager initialized
  Config     : F_GSP_optimization.yaml
  Parameters : 10

Parameters to optimize:
  initial_conditions.C_Glc_init: bounds=(1, 100)
  initial_conditions.C_Gln_init: bounds=(1, 100)
  initial_conditions.C_AA_init: bounds=(1, 100)
  initial_conditions.osm_bg_init: bounds=(100, 300)
  fed_batch.C_Glc_feed: bounds=(1, 200)


In [12]:
# ── Run optimization ───────────────────────────────────────────────────────────
opt_manager.optimize_parameters(
    objectives=OBJECTIVES,
    weights=WEIGHTS,
    workers=1,
    seed=SEED,
    timeout=TIMEOUT,
    verbose=True,
    normalization_values=campaign_norm,
    debug_rejections=True,
)

print("\n✓ Optimization completed")
print("\nOptimal parameters found:")
for param_name, value in opt_manager.optimal_parameters.items():
    cfg = opt_manager.opt_config[param_name]
    print(f"  {param_name}: {value:.6g} {cfg['unit']}")

Starting optimization with objectives: STY_kg_m3_day (weight: 10.0), medium_use_efficiency_kg_m3 (weight: 1.0), nutrient_use_efficiency_kg_kg (weight: 1.0)
Worker configuration: requested=1, effective=1
Optimizing 10 parameters: ['initial_conditions.C_Glc_init', 'initial_conditions.C_Gln_init', 'initial_conditions.C_AA_init', 'initial_conditions.osm_bg_init', 'fed_batch.C_Glc_feed', 'fed_batch.C_Gln_feed', 'fed_batch.C_AA_feed', 'fed_batch.C_Glc_setpoint', 'fed_batch.C_Glc_batch_trigger', 'fed_batch.osm_bg_feed']
Using campaign-level normalization scalars:
  STY_kg_m3_day: 0.7667
  medium_use_efficiency_kg_m3: 1.059
  nutrient_use_efficiency_kg_kg: 0.1535
Running differential evolution...
run_simulation execution time: 1582.277 ms
[ObjectiveFunction][reject][mode=FEDBATCH] incomplete_batch_final_volume: min_final_volume_fraction=0.30002450653374746; max_final_volume_fraction=0.4384716933190466; last_final_volume_fraction=0.30002450653374746; threshold=0.98
run_simulation execution time

KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# 5. Performance Comparison
# ==============================================================================
comparison_df = opt_manager.compare_performance(verbose=True)
comparison_df


=== PERFORMANCE COMPARISON ===
                         Metric                                                                      Description  Baseline      Unit  Optimized  Improvement
          Medium Use Efficiency               Dry biomass harvested per total medium volume sent to the process.  0.400056     kg/m3   0.420017     0.019962
Nutrient Consumption Efficiency             Dry biomass harvested per net nutrient mass consumed in the reactor.  0.193097     kg/kg   0.193045    -0.000052
        Nutrient Use Efficiency                Dry biomass harvested per total nutrient mass fed to the reactor.  0.041557     kg/kg   0.126013     0.084456
         Protein Use Efficiency                   Biomass protein harvested per proteinaceous nutrient mass fed.  0.032136     kg/kg   0.165289     0.133154
                            STY Space-time yield based on harvested dry biomass, reactor size, and elapsed time.  0.073379 kg/m3/day   0.072711    -0.000668
             batch_time_ra

,Metric,Description,Baseline,Unit,Optimized,Improvement
0,Medium Use Efficiency,Dry biomass harvested per total medium volume ...,0.400056,kg/m3,0.420017,0.019962
1,Nutrient Consumption Efficiency,Dry biomass harvested per net nutrient mass co...,0.193097,kg/kg,0.193045,-0.000052
2,Nutrient Use Efficiency,Dry biomass harvested per total nutrient mass ...,0.041557,kg/kg,0.126013,0.084456
3,Protein Use Efficiency,Biomass protein harvested per proteinaceous nu...,0.032136,kg/kg,0.165289,0.133154
4,STY,Space-time yield based on harvested dry biomas...,0.073379,kg/m3/day,0.072711,-0.000668
5,batch_time_range_h,,9.604990,,10.894539,1.289549


In [ ]:
# ==============================================================================
# 6. Final Simulation Check
# ==============================================================================
final_sim = opt_manager.optimized_simulation

print("\nFinal Performance Metrics:")
for key, value in final_sim.data.summary_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")


Final Performance Metrics:
  STY_kg_m3_day: 0.0727
  medium_use_efficiency_kg_m3: 0.4200
  protein_use_efficiency_kg_kg: 0.1653
  nutrient_use_efficiency_kg_kg: 0.1260
  nutrient_consumption_efficiency_kg_kg: 0.1930
  batch_time_range_h: 10.8945


In [ ]:
# ── Optimized simulation visualizations ────────────────────────────────────────
final_sim.data.plot_kinetics()

In [ ]:
final_sim.data.plot_stoichiometric_yields()

In [ ]:
final_sim.data.plot_volume_feed()

In [ ]:
fig_list = final_sim.data.plot_profiles()
for fig in fig_list:
    fig.show()

In [ ]:
# ==============================================================================
# 7. Export Results
#
# Three outputs saved to the local export/ directory:
#   c4_{SIM_ID}_opt_{timestamp}.xlsx         — full Excel export
#   c4_{SIM_ID}_opt_{timestamp}.pkl          — complete simulation state (pickle)
#   c4_{SIM_ID}_opt_{timestamp}.parameters.yaml — optimized parameter set
# ==============================================================================
EXPORT_DIR = SCRIPT_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# final_sim
final_sim.save(
    filename=EXPORT_FILENAME,
    file_format="excel",
    include_metadata=True,
    include_summary_stats=True,
    include_cultivation_summary=True,
    include_batch_data=True,
    include_parameters=False,
    include_pickle=True,
    include_opt_parameters=True,
    overwrite=True,
    export_dir=EXPORT_DIR,
)

print(f"\n✓ Export complete")
print(f"  Directory : {EXPORT_DIR}")
print("  Files written:")
for f in sorted(EXPORT_DIR.rglob("*")):
    if f.is_file():
        print(f"    {f.relative_to(EXPORT_DIR)}  ({f.stat().st_size / 1024:.1f} KB)")

Simulation data exported to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746.xlsx
CellCultureSimulation pickle saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746.pkl
Parameters saved to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export\c4_fedbatch_constant_opt_20260707_1746\c4_fedbatch_constant_opt_20260707_1746_opt_params.yaml

✓ Export complete
  Directory : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics_updated\export
  Files written:
    c4_fedbatch_constant_opt_20260629_1522\c4_fedbatch